# IIC3745 - Actividad 4: Symbolic Fuzzing con LLM

Un fuzzer simbolico hace dos cosas: **extrae** las condiciones de camino del
codigo, y luego las **resuelve** para obtener inputs concretos. En esta
actividad le pides al modelo que haga ambos pasos, por separado, y mides en
cual de los dos se rompe.

Separarlos importa: si le pides todo junto y falla, no sabes si no entendio el
codigo o si no supo resolver las restricciones. Midiendo por partes se puede
atribuir.

**La matriz:**

| | Resuelve Z3 | Resuelve el LLM |
|---|---|---|
| **Extrae el fuzzer simbolico** | **A** referencia | **D** |
| **Extrae el LLM** | **C** | **B** |

- **A** es el fuzzer simbolico completo, automatico. Es la referencia.
- **B** es el fuzzer simbolico hecho 100% por el modelo.
- **C** aisla la extraccion: si C alcanza mucho mas que B, el modelo sabia
  cuales eran las restricciones pero no supo resolverlas.
- **D** aisla lo contrario: restricciones correctas de entrada, resueltas por
  el modelo.

**Tres chats:** uno para extraer, uno para resolver sus propias restricciones,
y uno para resolver las del fuzzer.

In [1]:
# ============================================================
# CELDA SELLADA - NO MODIFICAR
# ============================================================
import os, sys, subprocess

# --- Google Colab -------------------------------------------------------
# !unzip -q -o Actividad4.zip -d actividad4
# RAIZ = "actividad4"
RAIZ = "."

if os.path.abspath(RAIZ) != os.getcwd():
    os.chdir(RAIZ)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"],
                   check=True)
    import z3

# fuzzingbook es OPCIONAL. Si lo instalas, el notebook lo usa como motor de
# caminos; si no, usa un enumerador propio que da exactamente los mismos
# caminos (verificado con Z3). Descomenta si quieres usarlo:
#
# !apt-get -qq update && apt-get -qq install -y graphviz graphviz-dev
# !pip install -q pygraphviz fuzzingbook pyan3 "setuptools<81"

print("z3", z3.get_version_string(), "| python", sys.version.split()[0])
print("directorio:", os.getcwd())

z3 5.3.0 | python 3.10.12
directorio: /home/moonse/TESTING/4-testing


In [2]:
# ============================================================
# CELDA SELLADA - NO MODIFICAR
# ============================================================
"""Motor de la Actividad 4 - Symbolic Fuzzing con LLM.

Tres piezas:

  1. Caminos de referencia. Los obtiene de fuzzingbook si esta disponible, y si
     no, de un enumerador propio equivalente para el estilo restringido de
     src/funciones.py. Ambos producen lo mismo.

  2. Traduccion de restricciones a Z3. Tolera como escriba el modelo la
     negacion y los operadores.

  3. Evaluacion. Para cada configuracion mide: caminos enumerados, equivalencia
     de restricciones, y que camino alcanza realmente cada input al ejecutarse.
"""

import ast
import contextlib
import importlib
import inspect
import json
import os
import re
import sys

import z3


RUTA_SUT = "src/funciones.py"
FUNCIONES = ["apply_coupon", "tarifa_noche", "puntaje_riesgo"]

RESULTADOS = {}
_MOTOR_CAMINOS = None


# ==========================================================================
# 1. Caminos de referencia
# ==========================================================================

def _enumerar_ast(fn):
    """Enumera los caminos de una funcion con ifs anidados y condiciones
    simples. Devuelve [[(condicion_fuente, tomada), ...], ...] en el orden en
    que aparecen, igual que fuzzingbook."""
    fuente = inspect.getsource(fn)
    arbol = ast.parse(fuente).body[0]

    def recorrer(cuerpo, acumulado):
        for nodo in cuerpo:
            if isinstance(nodo, ast.If):
                cond = ast.unparse(nodo.test)
                return (recorrer(nodo.body, acumulado + [(cond, True)])
                        + recorrer(nodo.orelse, acumulado + [(cond, False)]))
        return [acumulado]

    return recorrer(arbol.body, [])


@contextlib.contextmanager
def _silencio():
    """Silencia salida y errores incluso de subprocesos: fuzzingbook lanza pip
    al importarse y escribe directo al descriptor."""
    nulo = os.open(os.devnull, os.O_RDWR)
    guardados = (os.dup(1), os.dup(2))
    try:
        os.dup2(nulo, 1)
        os.dup2(nulo, 2)
        yield
    finally:
        os.dup2(guardados[0], 1)
        os.dup2(guardados[1], 2)
        for fd in guardados:
            os.close(fd)
        os.close(nulo)


def _caminos_fuzzingbook(fn):
    """Intenta usar fuzzingbook. Devuelve None si no esta disponible o falla.

    El import se intenta solo si el paquete existe: fuzzingbook ejecuta pip al
    importarse y sin esta guarda ensucia la salida con errores de instalacion
    cuando no esta presente."""
    import importlib.util
    if importlib.util.find_spec("fuzzingbook") is None:
        return None
    try:
        with _silencio():
            from fuzzingbook.SymbolicFuzzer import SimpleSymbolicFuzzer
            import fuzzingbook.SymbolicFuzzer as sf
    except Exception:
        return None
    try:
        with _silencio():
            f = SimpleSymbolicFuzzer(fn)
            rutas = f.get_all_paths(f.fnenter)
            exec(sf.define_symbolic_vars(f.used_variables, ""), sf.__dict__)
        salida = []
        for p in rutas:
            crudas = f.extract_constraints(p)
            # descartamos las asignaciones (var == expr): solo nos interesan
            # las condiciones de los if
            conds = [c for c in crudas if not _es_asignacion(c, fn)]
            salida.append(conds)
        return salida
    except Exception:
        return None


def _es_asignacion(restriccion, fn):
    """True si la restriccion es la asignacion de la variable de resultado."""
    params = set(inspect.signature(fn).parameters)
    m = re.match(r"^\s*([A-Za-z_]\w*)\s*==", restriccion)
    return bool(m) and m.group(1) not in params


def _normalizar_cond(c):
    """Lleva una restriccion al estilo 'not (cond)' o 'cond'."""
    c = c.strip()
    m = re.match(r"^(?:z3\.)?Not\s*\((.*)\)\s*$", c, re.S)
    if m:
        return "not (%s)" % m.group(1).strip()
    return c


def caminos_referencia(fn):
    """Lista de caminos; cada uno es una lista de condiciones normalizadas."""
    global _MOTOR_CAMINOS
    fb = _caminos_fuzzingbook(fn)
    if fb is not None:
        _MOTOR_CAMINOS = "fuzzingbook"
        return [[_normalizar_cond(c) for c in p] for p in fb]
    _MOTOR_CAMINOS = "enumerador propio (fuzzingbook no disponible)"
    return [[(c if tomada else "not (%s)" % c) for c, tomada in p]
            for p in _enumerar_ast(fn)]


def motor_caminos():
    return _MOTOR_CAMINOS or "sin determinar"


# ==========================================================================
# 2. Restricciones -> Z3
# ==========================================================================

_REEMPLAZOS = [
    (r"(?<![\w.])z3\.Not\s*\(", "Not("),
    (r"(?<![\w.])Not\s*\(", "Not("),
    (r"(?<![\w.])NOT\s*\(", "Not("),
    (r"(?<![\w.])not\s+", "Not_PREFIJO "),
    (r"!\s*\(", "Not("),
    (r"(?<![=!<>])=(?![=])", "=="),
    (r"<>", "!="),
    (r"≥", ">="), (r"≤", "<="), (r"≠", "!="), (r"¬", "Not_PREFIJO "),
    (r"(?<![\w.])AND(?![\w])", "and"),
    (r"(?<![\w.])OR(?![\w])", "or"),
]


def _a_expresion(texto):
    """Normaliza como el modelo haya escrito la restriccion."""
    t = texto.strip().rstrip(";").strip()
    for patron, nuevo in _REEMPLAZOS:
        t = re.sub(patron, nuevo, t)
    # 'not (x > 1)' -> 'Not(x > 1)'
    t = re.sub(r"Not_PREFIJO\s*\((.*)\)\s*$", r"Not(\1)", t, flags=re.S)
    t = re.sub(r"Not_PREFIJO\s*(.+)$", r"Not(\1)", t, flags=re.S)
    return t


def a_z3(restricciones, variables):
    """Convierte una lista de restricciones en una conjuncion de Z3.

    variables: nombres de los parametros de la funcion.
    Devuelve (expresion, problemas)."""
    entorno = {v: z3.Int(v) for v in variables}
    entorno.update({"Not": z3.Not, "And": z3.And, "Or": z3.Or,
                    "Implies": z3.Implies, "z3": z3, "True": True, "False": False})
    partes, problemas = [], []
    for r in restricciones:
        if not isinstance(r, str):
            problemas.append("restriccion que no es texto: %r" % (r,))
            continue
        expr = _a_expresion(r)
        try:
            valor = eval(expr, {"__builtins__": {}}, entorno)
        except Exception as e:
            problemas.append("no se pudo interpretar %r (%s)" % (r, type(e).__name__))
            continue
        if isinstance(valor, bool):
            valor = z3.BoolVal(valor)
        partes.append(valor)
    if not partes:
        return None, problemas
    return (partes[0] if len(partes) == 1 else z3.And(*partes)), problemas


def resolver(restricciones, variables):
    """Resuelve con Z3. Devuelve (asignacion o None, problemas)."""
    expr, problemas = a_z3(restricciones, variables)
    if expr is None:
        return None, problemas
    s = z3.Solver()
    s.add(expr)
    if s.check() != z3.sat:
        return None, problemas
    m = s.model()
    return {v: m.evaluate(z3.Int(v), model_completion=True).as_long()
            for v in variables}, problemas


def equivalentes(a, b, variables):
    """True si dos conjuntos de restricciones son logicamente equivalentes."""
    ea, pa = a_z3(a, variables)
    eb, pb = a_z3(b, variables)
    if ea is None or eb is None:
        return False
    s = z3.Solver()
    s.add(z3.Distinct(z3.If(ea, 1, 0), z3.If(eb, 1, 0)))
    return s.check() == z3.unsat


# ==========================================================================
# 3. Que camino toma realmente un input
# ==========================================================================

def firma_de_ejecucion(fn, args):
    """Ejecuta la funcion y devuelve la tupla de lineas que recorrio."""
    lineas = []
    archivo = inspect.getsourcefile(fn)
    inicio = inspect.getsourcelines(fn)[1]
    fin = inicio + len(inspect.getsourcelines(fn)[0])

    def traza(frame, evento, arg):
        if evento == "line" and frame.f_code.co_filename == archivo \
                and inicio <= frame.f_lineno < fin:
            lineas.append(frame.f_lineno)
        return traza

    anterior = sys.gettrace()
    sys.settrace(traza)
    try:
        fn(*args)
    finally:
        sys.settrace(anterior)
    return tuple(lineas)


def firmas_de_referencia(fn, caminos, variables):
    """Para cada camino de referencia, resuelve y ejecuta para obtener su
    firma. Devuelve {firma: indice_de_camino}."""
    firmas = {}
    for i, c in enumerate(caminos):
        sol, _ = resolver(c, variables)
        if sol is None:
            continue
        args = [sol[v] for v in variables]
        firmas[firma_de_ejecucion(fn, args)] = i
    return firmas


# ==========================================================================
# 4. Prompts
# ==========================================================================

INSTRUCCION_P1 = """You are a symbolic execution engine. For EACH function below, \
enumerate ALL execution paths. For each path, give the list of path constraints: \
the condition of every `if` the path goes through, written as it must hold on \
that path.

Rules:
- Use the parameter names exactly as they appear in the code.
- Write a condition that is taken as-is, and one that is not taken as \
`not (condition)`.
- Do NOT include assignments, only the conditions of the `if` statements.
- One entry per path, in any order.

Answer with a JSON object and nothing else, mapping each function name to a \
list of paths, where each path is a list of constraint strings:

  {"nombre_funcion": [["a > 10", "b > 5"], ["a > 10", "not (b > 5)"]], ...}
"""

INSTRUCCION_P2 = """Below is a list of path constraints. For EACH path, give \
concrete integer values for the variables that satisfy ALL of its constraints \
at the same time. If a path is impossible, use null.

Answer with a JSON object and nothing else, with the same structure, where each \
path becomes an object of variable values:

  {"nombre_funcion": [{"a": 11, "b": 6}, {"a": 11, "b": 0}], ...}
"""


def _codigo_sut():
    return open(RUTA_SUT, encoding="utf-8").read()


def mostrar_prompt_paso1():
    print("=" * 78)
    print("PASO 1 - EXTRACCION. Copia todo esto en un chat nuevo.")
    print("=" * 78)
    print()
    print(INSTRUCCION_P1)
    print("```python")
    print(_codigo_sut().rstrip())
    print("```")
    print()
    print("=" * 78)
    print("Guarda la respuesta cruda en prompts/paso1.md y el JSON en")
    print("inputs/constraints_llm.json")
    print("=" * 78)


def _bloque_restricciones(fuente):
    """fuente: 'llm' o 'ref'. Devuelve el texto con las restricciones."""
    lineas = []
    for n in FUNCIONES:
        fn = _funcion(n)
        cam = (_caminos_llm().get(n, []) if fuente == "llm"
               else caminos_referencia(fn))
        lineas.append("%s (variables: %s)" % (n, ", ".join(_variables(fn))))
        for i, c in enumerate(cam):
            lineas.append("  path %d: %s" % (i, " AND ".join(c)))
        lineas.append("")
    return "\n".join(lineas)


def mostrar_prompt_paso2(fuente="llm"):
    """fuente='llm' usa las restricciones que el modelo extrajo;
    fuente='ref' usa las del fuzzer simbolico."""
    etiqueta = ("las que tu mismo extrajiste" if fuente == "llm"
                else "las del fuzzer simbolico")
    print("=" * 78)
    print("PASO 2 - RESOLUCION (%s). Copia todo esto en un chat NUEVO." % etiqueta)
    print("=" * 78)
    print()
    print(INSTRUCCION_P2)
    print(_bloque_restricciones(fuente))
    print("=" * 78)
    archivo = "solucion_llm.json" if fuente == "llm" else "solucion_llm_ref.json"
    print("Guarda la respuesta cruda en prompts/paso2_%s.md y el JSON en" % fuente)
    print("inputs/%s" % archivo)
    print("=" * 78)


# ==========================================================================
# 5. Lectura tolerante de las respuestas
# ==========================================================================

ENVOLTORIOS = ("solucion", "solution", "answer", "result", "respuesta",
               "paths", "caminos", "constraints", "restricciones", "data")


def _leer_json(ruta):
    texto = open(ruta, encoding="utf-8").read().strip()
    if not texto:
        raise ValueError("%s esta vacio" % ruta)
    if texto.startswith("```"):
        texto = "\n".join(l for l in texto.splitlines()
                          if not l.strip().startswith("```"))
    texto = re.sub(r"^\s*(//|#).*$", "", texto, flags=re.M)
    texto = re.sub(r",(\s*[\]\}])", r"\1", texto)
    try:
        return json.loads(texto)
    except json.JSONDecodeError as e:
        try:
            datos = ast.literal_eval(texto)
        except (ValueError, SyntaxError):
            raise ValueError("%s no es JSON valido ni un dict de Python (%s)"
                             % (ruta, e))
        print("  [aviso] %s venia como dict de Python. Se leyo igual." % ruta)
        return datos


def _desenvolver(v):
    n = 0
    while isinstance(v, dict) and len(v) == 1 and n < 4:
        k = next(iter(v))
        if str(k).strip().lower() not in ENVOLTORIOS:
            break
        v = v[k]
        n += 1
    return v


def _clave(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def _funcion(nombre):
    mod = importlib.import_module("src.funciones")
    importlib.reload(mod)
    return getattr(mod, nombre)


def _variables(fn):
    return list(inspect.signature(fn).parameters)


def _por_funcion(datos):
    """Mapea las llaves del modelo a los nombres reales de las funciones."""
    datos = _desenvolver(datos)
    reales = {_clave(f): f for f in FUNCIONES}
    salida, avisos, errores = {}, [], []
    if not isinstance(datos, dict):
        return {}, [], ["se esperaba un objeto {funcion: ...}, llego %s"
                        % type(datos).__name__]
    for k, v in datos.items():
        real = reales.get(_clave(k))
        if real is None:
            errores.append("no existe la funcion %r (validas: %s)"
                           % (k, ", ".join(FUNCIONES)))
            continue
        if k != real:
            avisos.append("%r se interpreto como %s" % (k, real))
        salida[real] = _desenvolver(v)
    return salida, avisos, errores


_CACHE_LLM = None


def _caminos_llm():
    global _CACHE_LLM
    if _CACHE_LLM is not None:
        return _CACHE_LLM
    datos, avisos, errores = _por_funcion(_leer_json("inputs/constraints_llm.json"))
    limpio = {}
    for n, caminos in datos.items():
        if isinstance(caminos, dict):
            caminos = list(caminos.values())
        if not isinstance(caminos, list):
            errores.append("%s: se esperaba una lista de caminos" % n)
            continue
        norm = []
        for p in caminos:
            if isinstance(p, str):
                p = [x.strip() for x in re.split(r"\bAND\b|&&", p) if x.strip()]
            if isinstance(p, dict):
                p = list(p.values())
            if not isinstance(p, list):
                errores.append("%s: un camino no es una lista (%r)" % (n, p))
                continue
            norm.append([str(x) for x in p])
        limpio[n] = norm
    for a in avisos:
        print("  [aviso] %s" % a)
    for e in errores:
        print("  [error] %s" % e)
    _CACHE_LLM = limpio
    return limpio


# ==========================================================================
# 6. Evaluacion
# ==========================================================================

def evaluar_extraccion(mostrar=True):
    """Compara los caminos que extrajo el modelo con los de referencia."""
    llm = _caminos_llm()
    filas = []
    for n in FUNCIONES:
        fn = _funcion(n)
        vs = _variables(fn)
        ref = caminos_referencia(fn)
        prop = llm.get(n, [])
        emparejados, sobrantes = set(), []
        for p in prop:
            calce = next((i for i in range(len(ref))
                          if i not in emparejados and equivalentes(p, ref[i], vs)),
                         None)
            if calce is None:
                sobrantes.append(p)
            else:
                emparejados.add(calce)
        faltantes = [ref[i] for i in range(len(ref)) if i not in emparejados]
        filas.append({"funcion": n, "ref": len(ref), "propuestos": len(prop),
                      "correctos": len(emparejados), "faltantes": faltantes,
                      "sobrantes": sobrantes, "variables": vs})
    RESULTADOS["extraccion"] = filas
    if mostrar:
        print("Paso 1 - extraccion de restricciones")
        print("-" * 68)
        for f in filas:
            print("  %-16s referencia %d | propuso %d | equivalentes %d"
                  % (f["funcion"], f["ref"], f["propuestos"], f["correctos"]))
            for p in f["faltantes"]:
                print("      camino NO encontrado: %s" % " AND ".join(p))
            for p in f["sobrantes"]:
                sat, _ = resolver(p, f["variables"])
                marca = "infactible" if sat is None else "no calza con ninguno"
                print("      camino de mas (%s): %s" % (marca, " AND ".join(p)))
        tot_ref = sum(f["ref"] for f in filas)
        tot_ok = sum(f["correctos"] for f in filas)
        print("-" * 68)
        print("  Caminos correctos: %d de %d" % (tot_ok, tot_ref))
    return filas


def _leer_soluciones(ruta):
    datos, avisos, errores = _por_funcion(_leer_json(ruta))
    for a in avisos:
        print("  [aviso] %s" % a)
    for e in errores:
        print("  [error] %s" % e)
    return datos


def _entero(v):
    if isinstance(v, bool):
        return None
    if isinstance(v, int):
        return v
    if isinstance(v, float):
        return int(v) if float(v).is_integer() else None
    if isinstance(v, str):
        try:
            return int(v.strip())
        except ValueError:
            return None
    return None


def _args_de(asignacion, variables):
    """Convierte la asignacion del modelo en una tupla de argumentos."""
    if asignacion is None:
        return None
    asignacion = _desenvolver(asignacion)
    if isinstance(asignacion, (list, tuple)):
        if len(asignacion) != len(variables):
            return None
        asignacion = dict(zip(variables, asignacion))
    if not isinstance(asignacion, dict):
        return None
    porminus = {str(k).strip().lower(): v for k, v in asignacion.items()}
    args = []
    for v in variables:
        if v.lower() not in porminus:
            return None
        n = _entero(porminus[v.lower()])
        if n is None:
            return None
        args.append(n)
    return args


def _evaluar_solucion(etiqueta, caminos_por_funcion, soluciones, mostrar=True):
    """Para cada camino, comprueba si la asignacion lo satisface y que camino
    alcanza de verdad al ejecutar."""
    filas = []
    for n in FUNCIONES:
        fn = _funcion(n)
        vs = _variables(fn)
        ref = caminos_referencia(fn)
        firmas = firmas_de_referencia(fn, ref, vs)
        caminos = caminos_por_funcion.get(n, [])
        props = soluciones.get(n, [])
        if isinstance(props, dict):
            props = list(props.values())
        satisface = alcanzados = sin_respuesta = 0
        vistos = set()
        detalles = []
        for i, c in enumerate(caminos):
            prop = props[i] if i < len(props) else None
            args = _args_de(prop, vs)
            if args is None:
                sin_respuesta += 1
                detalles.append((i, "sin asignacion valida"))
                continue
            expr, _ = a_z3(c, vs)
            ok = False
            if expr is not None:
                s = z3.Solver()
                s.add(expr)
                for v, val in zip(vs, args):
                    s.add(z3.Int(v) == val)
                ok = s.check() == z3.sat
            if ok:
                satisface += 1
            else:
                detalles.append((i, "no satisface sus restricciones"))
            f = firma_de_ejecucion(fn, args)
            if f in firmas and f not in vistos:
                vistos.add(f)
                alcanzados += 1
        filas.append({"funcion": n, "caminos": len(caminos),
                      "satisface": satisface, "sin_respuesta": sin_respuesta,
                      "alcanzados": alcanzados, "ref": len(ref),
                      "detalles": detalles})
    RESULTADOS[etiqueta] = filas
    if mostrar:
        print(etiqueta)
        print("-" * 68)
        for f in filas:
            print("  %-16s %d/%d satisfacen | alcanza %d de %d caminos reales"
                  % (f["funcion"], f["satisface"], f["caminos"],
                     f["alcanzados"], f["ref"]))
            for i, m in f["detalles"][:4]:
                print("      path %d: %s" % (i, m))
    return filas


def evaluar_B(mostrar=True):
    """LLM extrae + LLM resuelve."""
    return _evaluar_solucion("B: LLM extrae + LLM resuelve", _caminos_llm(),
                             _leer_soluciones("inputs/solucion_llm.json"), mostrar)


def evaluar_C(mostrar=True):
    """LLM extrae + Z3 resuelve."""
    llm = _caminos_llm()
    sol = {}
    for n, caminos in llm.items():
        vs = _variables(_funcion(n))
        sol[n] = [resolver(c, vs)[0] for c in caminos]
    return _evaluar_solucion("C: LLM extrae + Z3 resuelve", llm, sol, mostrar)


def evaluar_D(mostrar=True):
    """Fuzzer simbolico extrae + LLM resuelve."""
    ref = {n: caminos_referencia(_funcion(n)) for n in FUNCIONES}
    return _evaluar_solucion("D: fuzzer extrae + LLM resuelve", ref,
                             _leer_soluciones("inputs/solucion_llm_ref.json"),
                             mostrar)


def evaluar_A(mostrar=True):
    """Referencia: fuzzer simbolico extrae + Z3 resuelve."""
    ref = {n: caminos_referencia(_funcion(n)) for n in FUNCIONES}
    sol = {}
    for n, caminos in ref.items():
        vs = _variables(_funcion(n))
        sol[n] = [resolver(c, vs)[0] for c in caminos]
    return _evaluar_solucion("A: fuzzer extrae + Z3 resuelve (referencia)",
                             ref, sol, mostrar)


def consolidar():
    print("Motor de caminos:", motor_caminos())
    ext = RESULTADOS.get("extraccion")
    if ext:
        print("\n## Tabla 1 - Paso 1: extraccion\n")
        print("| Funcion | Caminos reales | Propuestos | Equivalentes | "
              "Faltantes | De mas |")
        print("|---|---|---|---|---|---|")
        for f in ext:
            print("| %s | %d | %d | %d | %d | %d |" % (
                f["funcion"], f["ref"], f["propuestos"], f["correctos"],
                len(f["faltantes"]), len(f["sobrantes"])))

    print("\n## Tabla 2 - Paso 2: resolucion (matriz 2x2)\n")
    print("| Config | Extrae | Resuelve | Satisfacen | Caminos alcanzados |")
    print("|---|---|---|---|---|")
    etiquetas = [
        ("A: fuzzer extrae + Z3 resuelve (referencia)", "fuzzer", "Z3"),
        ("B: LLM extrae + LLM resuelve", "LLM", "LLM"),
        ("C: LLM extrae + Z3 resuelve", "LLM", "Z3"),
        ("D: fuzzer extrae + LLM resuelve", "fuzzer", "LLM"),
    ]
    for clave, ex, re_ in etiquetas:
        filas = RESULTADOS.get(clave)
        letra = clave.split(":")[0]
        if filas is None:
            print("| %s | %s | %s | - | - |" % (letra, ex, re_))
            continue
        sat = sum(f["satisface"] for f in filas)
        tot = sum(f["caminos"] for f in filas)
        alc = sum(f["alcanzados"] for f in filas)
        ref = sum(f["ref"] for f in filas)
        print("| %s | %s | %s | %d/%d | %d/%d |" % (letra, ex, re_, sat, tot, alc, ref))

    print("\n" + "-" * 68)
    print("Lee la tabla asi: si C alcanza mucho mas que B, el cuello de botella")
    print("esta en RESOLVER. Si C tambien queda corto, el problema venia de")
    print("EXTRAER. Compara D con A para aislar lo mismo al reves.")
    print("Copia las dos tablas a REPORTE.md.")

## 0. Verificacion previa

El motor de caminos puede ser fuzzingbook o el enumerador propio incluido. Los
dos producen los mismos caminos para este codigo, verificado con Z3, asi que la
referencia es la misma en cualquier caso. La celda te dice cual esta usando.

In [3]:
for n in FUNCIONES:
    print(n, len(caminos_referencia(_funcion(n))), "caminos")
print("motor:", motor_caminos())

apply_coupon 2 caminos
tarifa_noche 4 caminos
puntaje_riesgo 8 caminos
motor: enumerador propio (fuzzingbook no disponible)


## 1. Paso 1 - Extraccion

Chat nuevo. El modelo debe enumerar todos los caminos de las tres funciones y
dar las condiciones de cada uno.

Guarda la respuesta cruda en `prompts/paso1.md` y el JSON en
`inputs/constraints_llm.json`.

In [4]:
mostrar_prompt_paso1()

PASO 1 - EXTRACCION. Copia todo esto en un chat nuevo.

You are a symbolic execution engine. For EACH function below, enumerate ALL execution paths. For each path, give the list of path constraints: the condition of every `if` the path goes through, written as it must hold on that path.

Rules:
- Use the parameter names exactly as they appear in the code.
- Write a condition that is taken as-is, and one that is not taken as `not (condition)`.
- Do NOT include assignments, only the conditions of the `if` statements.
- One entry per path, in any order.

Answer with a JSON object and nothing else, mapping each function name to a list of paths, where each path is a list of constraint strings:

  {"nombre_funcion": [["a > 10", "b > 5"], ["a > 10", "not (b > 5)"]], ...}

```python
# Tres funciones con caminos de ejecucion bien definidos.
#
# Todas estan escritas en el estilo que el fuzzer simbolico de fuzzingbook
# puede analizar: cada condicion es una comparacion simple, cada camino asigna


### Que tan bien extrajo

El notebook compara cada camino propuesto contra los reales usando **Z3**: dos
conjuntos de restricciones cuentan como el mismo camino si son logicamente
equivalentes, aunque esten escritos distinto. Tambien detecta caminos de mas,
y te dice si son infactibles.

In [5]:
evaluar_extraccion()

Paso 1 - extraccion de restricciones
--------------------------------------------------------------------
  apply_coupon     referencia 2 | propuso 2 | equivalentes 2
  tarifa_noche     referencia 4 | propuso 4 | equivalentes 4
  puntaje_riesgo   referencia 8 | propuso 8 | equivalentes 8
--------------------------------------------------------------------
  Caminos correctos: 14 de 14


[{'funcion': 'apply_coupon',
  'ref': 2,
  'propuestos': 2,
  'correctos': 2,
  'faltantes': [],
  'sobrantes': [],
  'variables': ['total', 'qty']},
 {'funcion': 'tarifa_noche',
  'ref': 4,
  'propuestos': 4,
  'correctos': 4,
  'faltantes': [],
  'sobrantes': [],
  'variables': ['dias', 'piso', 'edad']},
 {'funcion': 'puntaje_riesgo',
  'ref': 8,
  'propuestos': 8,
  'correctos': 8,
  'faltantes': [],
  'sobrantes': [],
  'variables': ['monto', 'mora', 'avales', 'historial']}]

## 2. Paso 2 - Resolucion

Dos chats mas, los dos nuevos:

1. `mostrar_prompt_paso2("llm")` le pide resolver **sus propias** restricciones.
   Guarda en `inputs/solucion_llm.json`.
2. `mostrar_prompt_paso2("ref")` le pide resolver **las del fuzzer simbolico**.
   Guarda en `inputs/solucion_llm_ref.json`.

El segundo es el que permite aislar: si con restricciones correctas tampoco
acierta, el problema no era que no entendiera el codigo.

In [6]:
mostrar_prompt_paso2("llm")

PASO 2 - RESOLUCION (las que tu mismo extrajiste). Copia todo esto en un chat NUEVO.

Below is a list of path constraints. For EACH path, give concrete integer values for the variables that satisfy ALL of its constraints at the same time. If a path is impossible, use null.

Answer with a JSON object and nothing else, with the same structure, where each path becomes an object of variable values:

  {"nombre_funcion": [{"a": 11, "b": 6}, {"a": 11, "b": 0}], ...}

apply_coupon (variables: total, qty)
  path 0: qty >= 3
  path 1: not (qty >= 3)

tarifa_noche (variables: dias, piso, edad)
  path 0: dias > 7 AND piso > 10
  path 1: dias > 7 AND not (piso > 10)
  path 2: not (dias > 7) AND edad < 25
  path 3: not (dias > 7) AND not (edad < 25)

puntaje_riesgo (variables: monto, mora, avales, historial)
  path 0: monto > 500000 AND mora > 30 AND avales < 2
  path 1: monto > 500000 AND mora > 30 AND not (avales < 2)
  path 2: monto > 500000 AND not (mora > 30) AND historial < 12
  path 3: monto

In [7]:
mostrar_prompt_paso2("ref")

PASO 2 - RESOLUCION (las del fuzzer simbolico). Copia todo esto en un chat NUEVO.

Below is a list of path constraints. For EACH path, give concrete integer values for the variables that satisfy ALL of its constraints at the same time. If a path is impossible, use null.

Answer with a JSON object and nothing else, with the same structure, where each path becomes an object of variable values:

  {"nombre_funcion": [{"a": 11, "b": 6}, {"a": 11, "b": 0}], ...}

apply_coupon (variables: total, qty)
  path 0: qty >= 3
  path 1: not (qty >= 3)

tarifa_noche (variables: dias, piso, edad)
  path 0: dias > 7 AND piso > 10
  path 1: dias > 7 AND not (piso > 10)
  path 2: not (dias > 7) AND edad < 25
  path 3: not (dias > 7) AND not (edad < 25)

puntaje_riesgo (variables: monto, mora, avales, historial)
  path 0: monto > 500000 AND mora > 30 AND avales < 2
  path 1: monto > 500000 AND mora > 30 AND not (avales < 2)
  path 2: monto > 500000 AND not (mora > 30) AND historial < 12
  path 3: monto > 

### Las cuatro configuraciones

Para cada camino se comprueba dos cosas distintas:

- **Satisface**: Z3 verifica que la asignacion cumpla las restricciones de ese
  camino.
- **Alcanza**: se *ejecuta* la funcion con esos valores y se traza que lineas
  recorrio de verdad.

Pueden no coincidir, y ahi hay material para el reporte: una asignacion puede
satisfacer unas restricciones mal extraidas y aun asi no llegar al camino que
el modelo creia.

In [8]:
evaluar_A()

A: fuzzer extrae + Z3 resuelve (referencia)
--------------------------------------------------------------------
  apply_coupon     2/2 satisfacen | alcanza 2 de 2 caminos reales
  tarifa_noche     4/4 satisfacen | alcanza 4 de 4 caminos reales
  puntaje_riesgo   8/8 satisfacen | alcanza 8 de 8 caminos reales


[{'funcion': 'apply_coupon',
  'caminos': 2,
  'satisface': 2,
  'sin_respuesta': 0,
  'alcanzados': 2,
  'ref': 2,
  'detalles': []},
 {'funcion': 'tarifa_noche',
  'caminos': 4,
  'satisface': 4,
  'sin_respuesta': 0,
  'alcanzados': 4,
  'ref': 4,
  'detalles': []},
 {'funcion': 'puntaje_riesgo',
  'caminos': 8,
  'satisface': 8,
  'sin_respuesta': 0,
  'alcanzados': 8,
  'ref': 8,
  'detalles': []}]

In [9]:
evaluar_B()

B: LLM extrae + LLM resuelve
--------------------------------------------------------------------
  apply_coupon     2/2 satisfacen | alcanza 2 de 2 caminos reales
  tarifa_noche     4/4 satisfacen | alcanza 4 de 4 caminos reales
  puntaje_riesgo   8/8 satisfacen | alcanza 8 de 8 caminos reales


[{'funcion': 'apply_coupon',
  'caminos': 2,
  'satisface': 2,
  'sin_respuesta': 0,
  'alcanzados': 2,
  'ref': 2,
  'detalles': []},
 {'funcion': 'tarifa_noche',
  'caminos': 4,
  'satisface': 4,
  'sin_respuesta': 0,
  'alcanzados': 4,
  'ref': 4,
  'detalles': []},
 {'funcion': 'puntaje_riesgo',
  'caminos': 8,
  'satisface': 8,
  'sin_respuesta': 0,
  'alcanzados': 8,
  'ref': 8,
  'detalles': []}]

In [10]:
evaluar_C()

C: LLM extrae + Z3 resuelve
--------------------------------------------------------------------
  apply_coupon     2/2 satisfacen | alcanza 2 de 2 caminos reales
  tarifa_noche     4/4 satisfacen | alcanza 4 de 4 caminos reales
  puntaje_riesgo   8/8 satisfacen | alcanza 8 de 8 caminos reales


[{'funcion': 'apply_coupon',
  'caminos': 2,
  'satisface': 2,
  'sin_respuesta': 0,
  'alcanzados': 2,
  'ref': 2,
  'detalles': []},
 {'funcion': 'tarifa_noche',
  'caminos': 4,
  'satisface': 4,
  'sin_respuesta': 0,
  'alcanzados': 4,
  'ref': 4,
  'detalles': []},
 {'funcion': 'puntaje_riesgo',
  'caminos': 8,
  'satisface': 8,
  'sin_respuesta': 0,
  'alcanzados': 8,
  'ref': 8,
  'detalles': []}]

In [11]:
evaluar_D()

D: fuzzer extrae + LLM resuelve
--------------------------------------------------------------------
  apply_coupon     2/2 satisfacen | alcanza 2 de 2 caminos reales
  tarifa_noche     4/4 satisfacen | alcanza 4 de 4 caminos reales
  puntaje_riesgo   8/8 satisfacen | alcanza 8 de 8 caminos reales


[{'funcion': 'apply_coupon',
  'caminos': 2,
  'satisface': 2,
  'sin_respuesta': 0,
  'alcanzados': 2,
  'ref': 2,
  'detalles': []},
 {'funcion': 'tarifa_noche',
  'caminos': 4,
  'satisface': 4,
  'sin_respuesta': 0,
  'alcanzados': 4,
  'ref': 4,
  'detalles': []},
 {'funcion': 'puntaje_riesgo',
  'caminos': 8,
  'satisface': 8,
  'sin_respuesta': 0,
  'alcanzados': 8,
  'ref': 8,
  'detalles': []}]

## 3. Consolidacion

**CELDA SELLADA - NO MODIFICAR.**

In [12]:
# ============================================================
# CELDA SELLADA - NO MODIFICAR
# ============================================================
consolidar()

Motor de caminos: enumerador propio (fuzzingbook no disponible)

## Tabla 1 - Paso 1: extraccion

| Funcion | Caminos reales | Propuestos | Equivalentes | Faltantes | De mas |
|---|---|---|---|---|---|
| apply_coupon | 2 | 2 | 2 | 0 | 0 |
| tarifa_noche | 4 | 4 | 4 | 0 | 0 |
| puntaje_riesgo | 8 | 8 | 8 | 0 | 0 |

## Tabla 2 - Paso 2: resolucion (matriz 2x2)

| Config | Extrae | Resuelve | Satisfacen | Caminos alcanzados |
|---|---|---|---|---|
| A | fuzzer | Z3 | 14/14 | 14/14 |
| B | LLM | LLM | 14/14 | 14/14 |
| C | LLM | Z3 | 14/14 | 14/14 |
| D | fuzzer | LLM | 14/14 | 14/14 |

--------------------------------------------------------------------
Lee la tabla asi: si C alcanza mucho mas que B, el cuello de botella
esta en RESOLVER. Si C tambien queda corto, el problema venia de
EXTRAER. Compara D con A para aislar lo mismo al reves.
Copia las dos tablas a REPORTE.md.
